# Diploma Thesis: Introduction

The objective of this notebook is to introduce you to the development process of Diploma thesis: Compressing Large Language Models via Replacement of MLP Blocks.

The first objective of the thesis is to understand the data processing and parsing of NLP datasets, in this notebook we will prepare datasets C4 (training) + Wikitext  
(validation) and how to work with them correctly.

The main point is to create a MVP pipeline suitable for working and understanding the project concept. 

The MVP pipeline showcases a proof-of-concept replacement example of a **single Block**.

In [2]:
import os
import sys
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
import pprint

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [1]:
project_pwd = Path().cwd().parents[1]
project_pwd = os.path.abspath(project_pwd)
if project_pwd not in sys.path:
    sys.path.append(project_pwd)

sys.dont_write_bytecode = True

NameError: name 'Path' is not defined

In [3]:
from scripts.intro.model import *
from scripts.intro.loader import *
from scripts.intro.block import *
from scripts.intro.activation import *
from scripts.intro.replacement import *
from scripts.intro.eval import *
from scripts.intro.metrics import *

Model config

In [4]:
@dataclass
class CFG:
    model_id = "HuggingFaceTB/SmolLM2-1.7B"
    target_layer_idx = 3
    seq_len = 128
    batch_size = 2
    calib_split = "train[:5%]"
    eval_split = "validation[:1%]"
    num_calib_batches = 24
    num_eval_batches = 24
    replacement_epochs = 5
    replacement_lr = 1e-3
    replacement_batch_size = 2048
    seed = 21

In [5]:
cfg = CFG()
torch.manual_seed(cfg.seed)

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

mvp pipeline - block level

In [7]:
model, tokenizer = load_model_and_tokenizer(cfg.model_id, dtype=DTYPE, device=DEVICE)
calib_loader = make_loader(tokenizer, cfg.calib_split, cfg.seq_len, cfg.batch_size)
eval_loader = make_loader(tokenizer, cfg.eval_split, cfg.seq_len, cfg.batch_size)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [8]:
target_path = choose_target_mlp_path(model, cfg.target_layer_idx)
print(f"Target MLP path: {target_path}")

state_before = param_summary(model, target_path)

Candidate MLP paths:
   model.layers.0.mlp
   model.layers.1.mlp
   model.layers.2.mlp
   model.layers.3.mlp
   model.layers.4.mlp
   model.layers.5.mlp
   model.layers.6.mlp
   model.layers.7.mlp
   model.layers.8.mlp
   model.layers.9.mlp
Target MLP path: model.layers.3.mlp


baseline eval

In [9]:
base_loss, base_ppl = evaluate_lm(model, eval_loader, cfg.num_eval_batches, device=DEVICE)
print(f"[baseline] lm_loss={base_loss:.6f} ppl={base_ppl:.3f}")

[baseline] lm_loss=2.718328 ppl=15.155


calibration data collection (block=3 bcs. cfg.target_layer_idx=3), doing the same for validation calib. data

In [10]:
X_train, Y_train = collect_block_io(model, target_path, calib_loader, cfg.num_calib_batches, device=DEVICE)
print(f"Collected calibration pairs: X={tuple(X_train.shape)}, Y={tuple(Y_train.shape)}")

X_val, Y_val = collect_block_io(model, target_path, eval_loader, min(8, cfg.num_eval_batches), device=DEVICE)
print(f"Collected held-out pairs: X_val={tuple(X_val.shape)}, Y_val={tuple(Y_val.shape)}")

Collected calibration pairs: X=(6144, 2048), Y=(6144, 2048)
Collected held-out pairs: X_val=(2048, 2048), Y_val=(2048, 2048)


training replacement (e.g. simple linear layer)

In [11]:
hidden_size = X_train.shape[-1]
replacement = fit_linear_replacement(X_train, Y_train, hidden_size, device=DEVICE, cfg=cfg)
block_mse = evaluate_block_mse(replacement, X_val, Y_val, device=DEVICE)
print(f"[block validation] mse={block_mse:.6f}")

[replacement train] epoch=1 mse=0.724159
[replacement train] epoch=2 mse=0.651793
[replacement train] epoch=3 mse=0.601771
[replacement train] epoch=4 mse=0.566411
[replacement train] epoch=5 mse=0.540755
[block validation] mse=0.572931


replacement of original block

In [12]:
replace_submodule(model, target_path, replacement.to(DEVICE))

state_after = param_summary(model, target_path)

re-evaluation of the (full) model after replacement

In [13]:
swapped_loss, swapped_ppl = evaluate_lm(model, eval_loader, cfg.num_eval_batches, device=DEVICE)
print(f"[after swap] lm_loss={swapped_loss:.6f} ppl={swapped_ppl:.3f}")

[after swap] lm_loss=2.873085 ppl=17.692


In [ ]:
model_delta = (state_after["total_params"] - state_before["total_params"]) / state_before["total_params"]
block_delta = (state_after["target_block_params"] - state_before["target_block_params"]) / state_before["target_block_params"]

In [17]:
summary = {
        "target_path": target_path,
        "baseline_lm_loss": base_loss,
        "baseline_ppl": base_ppl,
        "block_val_mse": block_mse,
        "swapped_lm_loss": swapped_loss,
        "swapped_ppl": swapped_ppl,

        "delta_params_model_rltv" : model_delta / state_before["total_params"],
        "delta_params_block_rtv" : block_delta / state_before["target_block_params"]
    }

print("Summary:")
pprint.pprint(summary)

Summary:
{'baseline_lm_loss': 2.718327760696411,
 'baseline_ppl': np.float64(15.154958326635898),
 'block_val_mse': 0.5729309320449829,
 'delta_params_block_rtv': -1.8212530348036022e-08,
 'delta_params_model_rltv': -1.5752936428475385e-11,
 'swapped_lm_loss': 2.8730850219726562,
 'swapped_ppl': np.float64(17.691512803820903),
 'target_path': 'model.layers.3.mlp'}


___
Other useful stuff:

- Block metrics:
    - BI (Block Importance) score
    - cosine similarity score